# Domain Shift Variance Analysis

This notebook analyzes which layers in the Mamba-Vision backbone experience the highest domain shift between BDD-Day and BDD-Night (or ACDC). This helps identify which layers should be targeted for LoRA adapter injection.

In [ ]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import json
import numpy as np
from collections import defaultdict
import torchvision.transforms as transforms

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
from mamba_vision_ours.model import MambaVisionOurs
from pipelines.training import load_checkpoint, resolve_device

model = MambaVisionOurs(
    device=str(device),
    model_type="mamba_vision_T2",
    num_output_classes=8,
    pretrained=False,
)

base_checkpoint_path = REPO_ROOT / "checkpoints" / "base" / "coco_base.ckpt"
if base_checkpoint_path.exists():
    load_checkpoint(base_checkpoint_path, model)
    print(f"Loaded base checkpoint from {base_checkpoint_path}")
else:
    print(f"Warning: Base checkpoint not found at {base_checkpoint_path}")
    print("Proceeding with randomly initialized weights")

model = model.to(device)
model.eval()
print(f"Model moved to {device} and set to eval mode")

In [ ]:
class VarianceHook:
    def __init__(self):
        self.variances = {}
        self.handles = []

    def hook_fn(self, name):
        def fn(module, input, output):
            if isinstance(output, torch.Tensor):
                batch_size = output.shape[0]
                if len(output.shape) > 1:
                    flattened = output.view(batch_size, -1)
                    var = torch.var(flattened, dim=0).mean().item()
                    self.variances[name] = var
        return fn

    def register_hooks(self, model):
        for name, module in model.backbone.named_modules():
            if isinstance(module, nn.Linear):
                handle = module.register_forward_hook(self.hook_fn(name))
                self.handles.append(handle)

    def clear(self):
        for handle in self.handles:
            handle.remove()
        self.handles = []
        self.variances = {}

hook_obj = VarianceHook()
hook_obj.register_hooks(model)
print(f"Registered hooks on {len(hook_obj.handles)} linear layers in backbone")

In [ ]:
class COCOImageDataset(Dataset):
    def __init__(self, manifest_path, images_dir, max_samples=16, image_size=640):
        self.images_dir = Path(images_dir)
        self.image_size = image_size
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
        ])
        
        with open(manifest_path) as f:
            manifest = json.load(f)
        self.image_files = [img["file_name"] for img in manifest.get("images", [])][:max_samples]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.images_dir / self.image_files[idx]
        img = Image.open(img_path).convert("RGB")
        return self.transform(img)

bdd_day_manifest = REPO_ROOT / "configs" / "manifests" / "bdd_day_train.json"
bdd_day_images = REPO_ROOT / "data" / "exports" / "bdd_day" / "train" / "data"

bdd_night_manifest = REPO_ROOT / "configs" / "manifests" / "bdd_night_train.json"
bdd_night_images = REPO_ROOT / "data" / "exports" / "bdd_night" / "train" / "data"

if bdd_day_manifest.exists() and bdd_day_images.exists():
    day_dataset = COCOImageDataset(bdd_day_manifest, bdd_day_images, max_samples=16)
    day_loader = DataLoader(day_dataset, batch_size=16, shuffle=False)
    print(f"Loaded BDD Day dataset: {len(day_dataset)} images")
else:
    print(f"BDD Day dataset not found. Checked: {bdd_day_manifest}, {bdd_day_images}")
    day_loader = None

if bdd_night_manifest.exists() and bdd_night_images.exists():
    night_dataset = COCOImageDataset(bdd_night_manifest, bdd_night_images, max_samples=16)
    night_loader = DataLoader(night_dataset, batch_size=16, shuffle=False)
    print(f"Loaded BDD Night dataset: {len(night_dataset)} images")
else:
    print(f"BDD Night dataset not found. Checked: {bdd_night_manifest}, {bdd_night_images}")
    night_loader = None

In [ ]:
def compute_variances(loader, model, hook_obj, domain_name):
    hook_obj.clear()
    hook_obj.register_hooks(model)
    
    with torch.no_grad():
        for batch_idx, images in enumerate(loader):
            images = images.to(device)
            _ = model(images)
            if batch_idx >= 0:
                break
    
    variances = hook_obj.variances.copy()
    hook_obj.clear()
    print(f"{domain_name}: Collected variances from {len(variances)} layers")
    return variances

if day_loader is not None and night_loader is not None:
    print("Computing variance for BDD Day...")
    day_variances = compute_variances(day_loader, model, hook_obj, "BDD Day")
    
    print("Computing variance for BDD Night...")
    night_variances = compute_variances(night_loader, model, hook_obj, "BDD Night")
    
    variance_diffs = {}
    for layer_name in day_variances.keys():
        if layer_name in night_variances:
            diff = abs(day_variances[layer_name] - night_variances[layer_name])
            variance_diffs[layer_name] = diff
    
    sorted_diffs = sorted(variance_diffs.items(), key=lambda x: x[1], reverse=True)
    print(f"\nTotal layers with variance differences: {len(sorted_diffs)}")
else:
    print("Cannot compute variances: datasets not loaded")

In [ ]:
if variance_diffs:
    print("\n" + "="*80)
    print("TOP 10 LAYERS WITH HIGHEST DOMAIN SHIFT (BDD Day vs BDD Night)")
    print("="*80)
    print(f"{'Rank':<6} {'Layer Name':<50} {'Variance Diff':<15}")
    print("-"*80)
    
    for rank, (layer_name, diff) in enumerate(sorted_diffs[:10], start=1):
        print(f"{rank:<6} {layer_name:<50} {diff:.6f}")
    
    print("="*80)
    print("\nThese layers experience the highest domain shift and are good candidates")
    print("for LoRA adapter injection to improve model robustness across lighting conditions.")
else:
    print("No variance differences to display. Check dataset loading.")